# 04 — Modeling: Logistic Regression

We model 30-day readmission with a **logistic regression** — a generalized linear model with a binomial response (Ch 7.2). It is the natural choice here: the response is binary, and each effect reads as an **odds ratio**.

Plan:
- fit the full model and read off odds ratios, p-values, and confidence intervals
- compare it against an intercept-only (null) model and a reduced model using the **likelihood-ratio test, deviance, and AIC** (Ch 7.1)
- check in-sample classification accuracy against the majority-class baseline

**Inputs**: `data/processed/cleaned.csv`

In [1]:
# add the project folder to the path so we can import our own code in src/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
# statsmodels gives us the logistic-regression GLM with odds ratios, p-values, deviance and AIC
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

from src.config import PROCESSED_DIR, TARGET

In [3]:
# load the cleaned data and list the predictors by type
df = pd.read_csv(PROCESSED_DIR / 'cleaned.csv')

numeric = ['age', 'bmi', 'bnp', 'sodium', 'creatinine',
           'systolic_bp', 'heart_rate', 'adherence_score', 'distance_to_hospital_km']
binary = ['ace_inhibitor', 'beta_blocker', 'diuretic']
categorical = ['gender', 'income_level']
print('rows:', len(df))

rows: 3000


In [4]:
# standardize the numeric predictors (z-scores, Ch 2.3) so their odds ratios are comparable,
# then keep only complete rows so every model below is fit on the same data
model_df = df.copy()
model_df[numeric] = (model_df[numeric] - model_df[numeric].mean()) / model_df[numeric].std()

cols = [TARGET] + numeric + binary + categorical
model_df = model_df[cols].dropna()
print('complete-case rows:', len(model_df), 'of', len(df))

complete-case rows: 2720 of 3000


In [5]:
# fit the full logistic regression; categorical predictors enter as indicator variables via C()
predictors = numeric + binary + ['C(gender)', 'C(income_level)']
formula = f'{TARGET} ~ ' + ' + '.join(predictors)
full = smf.logit(formula, data=model_df).fit()
print(full.summary())

Optimization terminated successfully.
         Current function value: 0.627283
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:         readmitted_30d   No. Observations:                 2720
Model:                          Logit   Df Residuals:                     2704
Method:                           MLE   Df Model:                           15
Date:                Tue, 02 Jun 2026   Pseudo R-squ.:                 0.07654
Time:                        00:14:37   Log-Likelihood:                -1706.2
converged:                       True   LL-Null:                       -1847.6
Covariance Type:            nonrobust   LLR p-value:                 2.034e-51
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -0.1391      0.108     -1.284      0.199      -0.

In [6]:
# turn the coefficients into odds ratios with 95% CIs (easier to read than log-odds)
ci = np.exp(full.conf_int())
odds = pd.DataFrame({
    'odds_ratio': np.exp(full.params),
    'ci_low': ci[0],
    'ci_high': ci[1],
    'p_value': full.pvalues,
})
odds.sort_values('p_value').round(4)

,odds_ratio,ci_low,ci_high,p_value
adherence_score,0.6463,0.5953,0.7017,0.0000
bnp,1.4331,1.3206,1.5553,0.0000
ace_inhibitor,0.6847,0.5828,0.8044,0.0000
age,1.2067,1.1129,1.3083,0.0000
beta_blocker,0.7270,0.6189,0.8541,0.0001
distance_to_hospital_km,1.1479,1.0591,1.2442,0.0008
sodium,0.8712,0.8033,0.9449,0.0009
creatinine,1.1461,1.0558,1.2441,0.0011
systolic_bp,1.0757,0.9917,1.1667,0.0785
diuretic,1.1494,0.9782,1.3505,0.0906


## Comparing models

We compare the full model against two others:
- an **intercept-only (null) model** — do the predictors explain anything at all? (likelihood-ratio test)
- a **reduced model** with only the predictors that were significant in notebook 03 — does dropping the weak predictors hurt? (likelihood-ratio test + AIC)

In [7]:
# likelihood-ratio test of the full model vs an intercept-only model (are the predictors useful?)
null = smf.logit(f'{TARGET} ~ 1', data=model_df).fit()
lr_stat = 2 * (full.llf - null.llf)
lr_df = int(full.df_model)
print(f'LR chi2 = {lr_stat:.2f} on {lr_df} df, p = {stats.chi2.sf(lr_stat, lr_df):.3e}')
print(f'McFadden pseudo R2 = {full.prsquared:.4f}')
print(f'null deviance  = {-2 * null.llf:.1f}')
print(f'model deviance = {-2 * full.llf:.1f}')

Optimization terminated successfully.
         Current function value: 0.679276
         Iterations 4
LR chi2 = 282.84 on 15 df, p = 2.034e-51
McFadden pseudo R2 = 0.0765
null deviance  = 3695.3
model deviance = 3412.4


In [8]:
# reduced model: only the predictors that were significant earlier, compared back to the full model
reduced_terms = ['adherence_score', 'bnp', 'age', 'ace_inhibitor', 'beta_blocker', 'distance_to_hospital_km']
reduced = smf.logit(f'{TARGET} ~ ' + ' + '.join(reduced_terms), data=model_df).fit()

lr_stat = 2 * (full.llf - reduced.llf)
lr_df = int(full.df_model - reduced.df_model)
print(f'full vs reduced: LR chi2 = {lr_stat:.2f} on {lr_df} df, p = {stats.chi2.sf(lr_stat, lr_df):.3f}')
print(f'AIC  full = {full.aic:.1f}   reduced = {reduced.aic:.1f}')

Optimization terminated successfully.
         Current function value: 0.632550
         Iterations 5
full vs reduced: LR chi2 = 28.65 on 9 df, p = 0.001
AIC  full = 3444.4   reduced = 3455.1


In [9]:
# in-sample classification at a 0.5 cutoff: accuracy and a confusion table vs the majority baseline
pred = (full.predict(model_df) >= 0.5).astype(int)
accuracy = (pred == model_df[TARGET]).mean()
baseline = 1 - model_df[TARGET].mean()
print(f'model accuracy = {accuracy:.3f}')
print(f'baseline (always predict 0) = {baseline:.3f}')
pd.crosstab(model_df[TARGET], pred, rownames=['actual'], colnames=['predicted'])

model accuracy = 0.654
baseline (always predict 0) = 0.583


predicted,0,1
actual,,
0,1262,324
1,617,517


## Notes

- Odds ratios for the standardized numeric predictors are **per one standard deviation**; for the binary/indicator predictors they compare a level against its reference (e.g. taking an ACE inhibitor vs not).
- Likelihood-ratio tests and AIC are the model-comparison tools from Ch 7.1 — we deliberately do not use held-out cross-validation.
- This is complete-case analysis (rows with missing labs were dropped); imputation is outside the course scope.
- The fit is modest (low pseudo R-squared), consistent with the weak associations in notebook 03 — worth stating plainly in the report.

---
## Appendix (optional — beyond course scope)

The models below — decision tree, random forest, gradient boosting — and ROC-AUC evaluation are **not part of the AAI-500 syllabus**. They are included only as an informal machine-learning comparison and are not used for any conclusions in the report.

In [10]:
# optional ML comparison using the helpers in src/ (train/test split, ensembles, ROC-AUC) — beyond scope
from sklearn.model_selection import train_test_split
from src.config import RANDOM_STATE
from src.features.build import build_preprocessor
from src.models.train import candidate_models, make_pipeline
from src.models.evaluate import compare

X, y = df.drop(columns=[TARGET]), df[TARGET]
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes(exclude='number').columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

pre = build_preprocessor(num, cat)
fitted = {name: make_pipeline(pre, m).fit(X_tr, y_tr) for name, m in candidate_models().items()}
compare(fitted, X_te, y_te).round(3)

,accuracy,precision,recall,f1,roc_auc
logistic_regression,0.627,0.568,0.389,0.462,0.690
random_forest,0.625,0.564,0.393,0.463,0.674
gradient_boosting,0.605,0.527,0.397,0.453,0.664
decision_tree,0.585,0.496,0.530,0.513,0.577
baseline,0.588,0.000,0.000,0.000,0.500
